In [ ]:
import re
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from cycler import cycler

# ==============================
# Paper Figure Style Configuration
# ==============================
def get_transparent_color(color, transparency=0.5):
    """Convert a hex color to a lighter (more transparent) version."""
    c = mcolors.hex2color(color)
    c = [c[0] * transparency + (1.0 - transparency),
         c[1] * transparency + (1.0 - transparency),
         c[2] * transparency + (1.0 - transparency)]
    return "#{:02X}{:02X}{:02X}".format(int(c[0]*255), int(c[1]*255), int(c[2]*255))

palette = ['#1e90ff', '#ffbb00', '#ff5080', '#a7426d', '#ff3c10', "#282828"]
palette_facecolor = [get_transparent_color(c) for c in palette]
markers = ['o', 'P', '^', 's', 'p', 'h']
hatches = ['//', '\\\\', 'xx', '///', '\\\\\\', 'xxx']

plt.rcdefaults()
plt.rcParams["figure.figsize"] = [8, 4]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300

plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"

plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 8
plt.rcParams['lines.markeredgewidth'] = 2.0

plt.rcParams["font.size"] = 18
plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

plt.rcParams["legend.fontsize"] = "medium"
plt.rcParams["legend.facecolor"] = "white"
plt.rcParams["legend.edgecolor"] = "white"
plt.rcParams["legend.framealpha"] = 0.9
plt.rcParams['legend.frameon'] = False
plt.rcParams['legend.handlelength'] = 1.5
plt.rcParams['legend.handletextpad'] = 0.5
plt.rcParams['legend.columnspacing'] = 0.8
plt.rcParams['legend.labelspacing'] = 0.3

plt.rcParams["axes.prop_cycle"] = cycler(color=palette) + cycler(marker=markers)

# ==============================
# Load results from checkpoints
# ==============================
import json
from pathlib import Path

checkpoint_base = Path("../../logs/pose_estimation/real_world/checkpoints")

# Methods to load: wicompass, baseline, dance
methods = ["wicompass", "baseline", "dance"]

# Label mapping: dance -> IID
label_map = {
    "wicompass": "WiCompass",
    "baseline": "Baseline", 
    "dance": "Recollection"
}

def load_metrics_from_checkpoint(checkpoint_base: Path, method: str) -> dict:
    """Load best_metrics.json from checkpoint folder for given method."""
    # Find checkpoint folder matching the method
    pattern = f"PointTransformer-real_world_{method}_job*"
    matching_folders = list(checkpoint_base.glob(pattern))
    
    if not matching_folders:
        raise FileNotFoundError(f"No checkpoint found for method: {method}")
    
    # Use the most recent one (sorted by name, which includes timestamp)
    folder = sorted(matching_folders)[-1]
    metrics_file = folder / "best_metrics.json"
    
    with open(metrics_file) as f:
        metrics = json.load(f)
    
    return {
        'train_mpjpe': metrics['train']['mpjpe'],
        'test_mpjpe': metrics['test']['mpjpe'],
        'train_pmpjpe': metrics['train']['p_mpjpe'],
        'test_pmpjpe': metrics['test']['p_mpjpe'],
    }

# Load results for all methods
results = {}
for method in methods:
    results[method] = load_metrics_from_checkpoint(checkpoint_base, method)
    print(f"{label_map[method]}: Train MPJPE = {results[method]['train_mpjpe']:.2f}mm, "
          f"Test MPJPE = {results[method]['test_mpjpe']:.2f}mm")

In [ ]:
# ==============================
# Plot: Bar chart comparing Training vs Testing MPJPE
# ==============================
lw = 2

fig, ax = plt.subplots(figsize=(6, 5))

# Prepare data (reordered: IID (dance), WiCompass, Baseline)
methods_order = ["dance", "wicompass", "baseline"]
method_labels = [label_map[m] for m in methods_order]
train_mpjpes = [results[m]['train_mpjpe'] for m in methods_order]
test_mpjpes = [results[m]['test_mpjpe'] for m in methods_order]

x = np.arange(len(method_labels))
width = 0.35

# Draw bar chart with paper figure style (no error bars)
bars1 = ax.bar(x - width/2, train_mpjpes, width, 
               facecolor=palette_facecolor[0], edgecolor=palette[0],
               linewidth=lw, hatch=hatches[0], alpha=0.8,
               label='Training MPJPE')
bars2 = ax.bar(x + width/2, test_mpjpes, width, 
               facecolor=palette_facecolor[1], edgecolor=palette[1],
               linewidth=lw, hatch=hatches[1], alpha=0.8,
               label='Testing MPJPE')

# Add value labels on bars (larger font size)
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        label = f'{height:.1f}'
        ax.annotate(label,
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=18)

add_labels(bars1)
add_labels(bars2)

# Set chart properties
ax.set_ylabel('MPJPE (mm)')
# ax.set_xlabel('Method')
ax.set_xticks(x)
ax.set_xticklabels(method_labels)
ax.set_ylim(0, max(test_mpjpes) * 1.3)
ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5))

# Add grid
ax.grid(True, axis='y')
ax.set_axisbelow(True)

# Legend
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=2, frameon=False)

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.savefig('real_world_results.pdf', bbox_inches='tight', dpi=300)
plt.show()

print("\n✅ Figures saved to: experiments/real_world_scaling/real_world_results.pdf")

In [ ]:
# ==============================
# Plot: Testing MPJPE vs Data Size for all 12 experiments
# ==============================
def load_scaling_metrics(checkpoint_base: Path, config: str, sample_size: int) -> float:
    """Load test MPJPE for a specific config and sample size."""
    pattern = f"PointTransformer-real_world_{config}_{sample_size}_job*"
    matching_folders = list(checkpoint_base.glob(pattern))
    
    if not matching_folders:
        return None
    
    folder = sorted(matching_folders)[-1]
    metrics_file = folder / "best_metrics.json"
    
    if not metrics_file.exists():
        return None
    
    with open(metrics_file) as f:
        metrics = json.load(f)
    
    return metrics['test']['mpjpe']

# Load all 12 experiments
# Order: Recollection (dance), WiCompass, Baseline
configs = ["dance", "wicompass", "baseline"]
sample_sizes = [1000, 2000, 4000, 8000]

scaling_results = {}
for config in configs:
    scaling_results[config] = {}
    for size in sample_sizes:
        mpjpe = load_scaling_metrics(checkpoint_base, config, size)
        if mpjpe is not None:
            scaling_results[config][size] = mpjpe
            print(f"{label_map[config]} {size}: {mpjpe:.2f}mm")

# Plot grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(sample_sizes))
width = 0.25
lw = 2

for i, config in enumerate(configs):
    values = [scaling_results[config].get(size, 0) for size in sample_sizes]
    bars = ax.bar(x + i * width, values, width,
                  facecolor=palette_facecolor[i], edgecolor=palette[i],
                  linewidth=lw, hatch=hatches[i], alpha=0.8,
                  label=label_map[config])
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.annotate(f'{height:.1f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 5), textcoords="offset points",
                       ha='center', va='bottom', fontsize=14)

ax.set_ylabel('Testing MPJPE (mm)')
ax.set_xlabel('Training Samples')
ax.set_xticks(x + width)
ax.set_xticklabels([f'{s}' for s in sample_sizes])
ax.set_ylim(0, max([max(scaling_results[c].values()) for c in configs if scaling_results[c]]) * 1.2)
ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6))
ax.grid(True, axis='y')
ax.set_axisbelow(True)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3, frameon=False)

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.savefig('data_scaling_results.pdf', bbox_inches='tight', dpi=300)
plt.show()

print("\n✅ Figure saved to: experiments/real_world_scaling/data_scaling_results.pdf")
